[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yoyo4581/POKnowledgeBase_ETL/blob/main/Finetune_BioBERT_Colab.ipynb)

# Fine-tune BioBERT and build the Qdrant store

The one step of the vector-database pipeline that Airflow cannot run: it needs a GPU,
and Colab has no headless execution API, so a human opens this notebook.

Everything else is automated. `embedding_drift_check` (Airflow) is what tells you to come
here — it scores the model currently in production against the freshly held-out GO terms
and only raises the flag when `heldout_go_cosine_ndcg@10` drops below the floor or slides
against the previous run.

**What this notebook does**

1. Clones the repo (code only — the dataset and the weights never go through git).
2. Pulls the latest dataset release asset published by `embedding_dataset_export`.
3. Fine-tunes `dmis-lab/biobert-base-cased-v1.2` with the contrastive triplets.
4. Builds both Qdrant collections while the model is still on the GPU — free, since the
   encoder is already loaded.
5. Evaluates the result so you don't ship a bad run.
6. Zips the three artifacts in `/content` and copies them to Drive, where
   `qdrant_collection_build` picks them up.

**Runtime → Change runtime type → GPU** before running anything.

In [ ]:
#@title Runtime check
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> pick an A100 or L4. "
    "Training on CPU here would take days, not hours.")

# train.py loads the model in bfloat16 with flash-attn2 and unpadded inputs.
# All three need Ampere or newer (compute capability >= 8.0). A T4 is Turing
# (7.5) and will raise at model load -- which is the right outcome, because the
# fallback would be a run that looks fine and takes several times as long.
major, minor = torch.cuda.get_device_capability(0)
name = torch.cuda.get_device_name(0)
print(f"torch {torch.__version__} | {name} | compute capability {major}.{minor}")
assert major >= 8, (
    f"{name} is compute capability {major}.{minor}; bf16 + flash-attn2 need >= 8.0. "
    f"Runtime -> Change runtime type -> A100 or L4. (A T4 cannot run this config.)")
print("bf16 + flash-attn2 supported")

## 1. Configuration

`GITHUB_TOKEN` is only needed if the repo is private. Store it in Colab's secrets
(the key icon in the left sidebar) as `GITHUB_TOKEN` rather than pasting it into a cell —
a token in a notebook cell gets committed sooner or later.

In [ ]:
#@title Config
GITHUB_REPO   = "yoyo4581/POKnowledgeBase_ETL"  #@param {type:"string"}
GITHUB_BRANCH = "main"                          #@param {type:"string"}

# Where the artifacts land for Airflow to collect. Must match QDRANT_DRIVE_DIR
# on the ETL host (see dags/qdrant_build.py, ingest_artifacts).
DRIVE_DIR = "/content/drive/MyDrive/POKnowledgeBase/qdrant"  #@param {type:"string"}

# Chunking. These MUST match QDRANT_WINDOW / QDRANT_STRIDE in the ETL .env:
# the manifest records them, and a window covering a whole record collapses the
# chunk collection into the record collection.
WINDOW = 3  #@param {type:"integer"}
STRIDE = 2  #@param {type:"integer"}

# Training.
#   BATCH_SIZE            = in-batch negatives per anchor. Raising it does NOT
#                           make training faster -- same tokens, fewer optimizer
#                           steps. At ~35k rows, 256 gives ~137 steps/epoch and
#                           1024 only ~34, which undertrains at a fixed LR.
#   MINI_BATCH_NUM_TOKENS = the actual speed/VRAM dial. GradCache is exact at any
#                           chunk size, so raise this until you OOM. 65536 and
#                           131072 are the next rungs on an A100.
#   LEARNING_RATE         = does NOT auto-scale. If you do raise BATCH_SIZE, scale
#                           this with it (4x batch -> ~4e-5 sqrt / 8e-5 linear).
EPOCHS                = 10    #@param {type:"integer"}
BATCH_SIZE            = 256   #@param {type:"integer"}
MINI_BATCH_NUM_TOKENS = 32768 #@param {type:"integer"}
LEARNING_RATE         = 2e-5  #@param {type:"number"}
MAX_SEQ_LENGTH        = 512   #@param {type:"integer"}

REPO_DIR  = "/content/repo"
MODEL_DIR = "/content/biobert-go-retrieval"
STORE_DIR = "/content/qdrant_store"
EXPORT_DIR = "/content/qdrant_export"
EVAL_DIR  = "/content/eval_results"
# HF checkpoints are whole ~440MB model copies; keep them out of MODEL_DIR
# so the zip that ships to the ETL host is just the model.
CKPT_DIR  = "/content/ckpt"

# --- token, and prove it works before anything expensive depends on it -------
# This repo is private, so every read below needs a working credential. The
# failure modes look alike from four cells away, so they get separated here:
#   secret missing / not shared with this notebook -> no header sent -> 404
#   credential rejected by GitHub                  -> 401 Bad credentials
#   credential fine but no access to THIS repo     -> 404
GITHUB_TOKEN, _raw = "", None
try:
    from google.colab import userdata
    _raw = userdata.get("GITHUB_TOKEN")
except ImportError:
    print("Not on Colab; set GITHUB_TOKEN by hand if the repo is private.")
except Exception as e:
    # SecretNotFoundError -> no secret with that name
    # NotebookAccessError -> the secret exists but this notebook cannot read it
    print(f"Could not read the Colab secret: {type(e).__name__}\n"
          f"  Key icon in the left sidebar -> add GITHUB_TOKEN -> toggle\n"
          f"  'Notebook access' ON for THIS notebook.")

if _raw is not None:
    GITHUB_TOKEN = _raw.strip()
    # A trailing newline or space from the paste is the most common cause of a
    # 401 that looks like a correct token.
    if GITHUB_TOKEN != _raw:
        print(f"  note: stripped {len(_raw) - len(GITHUB_TOKEN)} whitespace "
              f"character(s) off the secret.")
    prefix = GITHUB_TOKEN.split("_")[0] + "_" if "_" in GITHUB_TOKEN else "(no prefix)"
    print(f"  secret read: {len(GITHUB_TOKEN)} chars, starts {prefix!r} "
          f"(expect 'github_pat_' fine-grained, or 'ghp_' classic)")

if GITHUB_TOKEN:
    import requests
    _h = {"Authorization": f"Bearer {GITHUB_TOKEN}",
          "Accept": "application/vnd.github+json",
          "X-GitHub-Api-Version": "2022-11-28"}

    who = requests.get("https://api.github.com/user", headers=_h, timeout=30)
    if who.status_code == 401:
        raise SystemExit(
            "GitHub rejected the token (401 Bad credentials). The token string "
            "itself is wrong -- truncated on paste, expired, or revoked. "
            "Regenerate at https://github.com/settings/personal-access-tokens "
            "and re-paste the WHOLE value into the Colab secret.")
    who.raise_for_status()
    print(f"  authenticated as {who.json()['login']}")

    repo = requests.get(f"https://api.github.com/repos/{GITHUB_REPO}",
                        headers=_h, timeout=30)
    if repo.status_code == 404:
        raise SystemExit(
            f"The token is valid but cannot see {GITHUB_REPO} (404). On a "
            f"fine-grained token this is 'Repository access' -- edit it, choose "
            f"'Only select repositories', tick this repo, and give it "
            f"'Contents: Read and write'.")
    repo.raise_for_status()
    info = repo.json()
    print(f"  {GITHUB_REPO}: private={info['private']}, "
          f"push={info['permissions']['push']} (push=True means Contents write)")
else:
    raise SystemExit(
        "No token, and this repo is private -- the clone and the dataset "
        "download below will both fail with 404. Add the GITHUB_TOKEN secret "
        "and re-run this cell.")

In [ ]:
#@title Mount Drive
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print("artifacts will be delivered to", DRIVE_DIR)

## 2. Code from GitHub, dataset from a release

Split on purpose. Code is small, diffable and belongs in git. `train.jsonl` is ~56MB and is
regenerated on every run of `embedding_dataset_export` — committing it would add that to the
repo's permanent history monthly, which no later `git rm` undoes. A release asset is
versioned, fetched with one GET, and adds nothing to clone size.

In [ ]:
#@title Clone
import os, shutil

# Step out of REPO_DIR before deleting it. On a re-run the kernel's cwd is still
# REPO_DIR from the previous run's `%cd`, and rmtree'ing the directory you are
# standing in leaves the kernel -- and every subprocess it spawns -- with no
# resolvable cwd. That is what "shell-init: error retrieving current directory"
# and git's "Unable to read current working directory" mean; the clone never
# even starts. chdir to an absolute path works even when the current cwd is
# already gone, so this also recovers a session that is in that state.
os.chdir("/content")
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

url = (f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git" if GITHUB_TOKEN
       else f"https://github.com/{GITHUB_REPO}.git")
!git clone --depth 1 --branch {GITHUB_BRANCH} {url} {REPO_DIR}

# A `!` cell does NOT raise when the command fails -- git prints its error and
# the notebook carries on. Check before cd'ing into it.
assert os.path.isdir(f"{REPO_DIR}/.git"), (
    f"Clone failed -- {REPO_DIR} has no .git. Read git's message above:\n"
    f"  'Authentication failed' / 'not found'       -> the token. Re-run the\n"
    f"     Config cell, which checks it against the API.\n"
    f"  'Unable to read current working directory'  -> the kernel's cwd was\n"
    f"     deleted. This cell now chdir's first, so just run it again.")

%cd {REPO_DIR}
!git log --oneline -1

for need in ("EmbeddingModel/BioBERT_Files/train.py",
             "src/builders/Qdrant/build/store.py",
             "src/builders/Qdrant/requirements.txt"):
    assert os.path.exists(f"{REPO_DIR}/{need}"), (
        f"{need} is missing from the clone. If it is store.py, the repo still "
        f"has the unanchored `build/` line in .gitignore swallowing it.")

# The training flags moved in e9c0b89. If the clone predates that, the Train
# cell below dies on an unrecognised argument several minutes from now.
_train = open(f"{REPO_DIR}/EmbeddingModel/BioBERT_Files/train.py").read()
assert "mini_batch_num_tokens" in _train, (
    "This clone has the old train.py (mini_batch_size, fp16, no flash-attn2). "
    "Push the port, or check GITHUB_BRANCH.")
print("clone OK -- train.py has the flash-attn2/bf16 port")

In [ ]:
#@title Install
# sentence-transformers 6 / transformers 5 -- train.py imports
# sentence_transformers.sentence_transformer.evaluation, which is the v6 layout.
!pip install -q "sentence-transformers[train]>=6" "transformers>=5" datasets accelerate
# attn_implementation="kernels-community/flash-attn2" resolves through the
# HF kernels hub, so the kernels package has to be present.
!pip install -q "kernels>=0.16.0,<0.17.0"
# qdrant-client, fastembed, mcp, numpy. fastembed is needed here too: BM25 is CPU-side
# and runs alongside the GPU dense pass.
!pip install -q -r src/builders/Qdrant/requirements.txt

In [ ]:
#@title Pull the latest dataset release
import io, json, os, tarfile, requests

api = f"https://api.github.com/repos/{GITHUB_REPO}"
headers = {"Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"}
if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

releases = requests.get(f"{api}/releases", headers=headers, params={"per_page": 50}, timeout=60)
releases.raise_for_status()
datasets = [r for r in releases.json() if r["tag_name"].startswith("dataset-")]
assert datasets, ("No dataset-* release found. Run the embedding_dataset_export DAG "
                  "first -- it builds the triplets from Neo4j + SQL and publishes them here.")

latest = datasets[0]
asset = next(a for a in latest["assets"] if a["name"].endswith(".tar.gz"))
print(f"{latest['tag_name']}  ->  {asset['name']}  ({asset['size']/1e6:.1f} MB)")
print(latest["body"])

# The asset API URL with octet-stream is what works for a private repo;
# browser_download_url would 404 without a session.
blob = requests.get(asset["url"], headers={**headers, "Accept": "application/octet-stream"},
                    timeout=900)
blob.raise_for_status()

os.makedirs("data/qdrant", exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(blob.content), mode="r:gz") as tar:
    tar.extractall("data/qdrant")

DATA_DIR = "data/qdrant/go_contrastive"
for name in sorted(os.listdir(DATA_DIR)):
    print(f"  {name:28s} {os.path.getsize(os.path.join(DATA_DIR, name))/1e6:8.2f} MB")

# Confirm the release was built with the Config in THIS clone. The release is
# whatever embedding_dataset_export last published -- if that DAG has not run
# since dataset.py changed, you would fine-tune on the old min_pos/max_pos and
# only find out by reading stats afterwards. Reads files only; no DB.
!python -m EmbeddingModel.BioBERT_Files.probe_config --verify-export {DATA_DIR}

## 3. Fine-tune

`--from-export` skips `extract()` / `build_dataset()` entirely, so nothing here imports
`Neo4jCaller` or `SQLCaller` — neither would import on a machine with no Neo4j credentials
and no ODBC driver, which Colab is.

Expect one to three hours on a T4 at these settings. The checkpoint that gets saved is
whichever epoch scored best on `heldout_go_cosine_ndcg@10`, not the last one — training loss
keeps dropping past the point held-out retrieval peaks.

In [ ]:
#@title Train
!python -m EmbeddingModel.BioBERT_Files.train \
    --from-export {DATA_DIR} \
    --out-dir {MODEL_DIR} \
    --epochs {EPOCHS} \
    --batch-size {BATCH_SIZE} \
    --mini-batch-size {MINI_BATCH_SIZE} \
    --learning-rate {LEARNING_RATE}

## 4. Build both collections

Done here rather than on the ETL host only because the encoder is already loaded on a GPU.
`eval_corpus.json` is, despite the name, the *whole* corpus — `build_dataset()` splits on GO
terms, not on proteins — so this one store serves both the experiment and production.

Two output folders because neither format does both jobs: `qdrant_store/` is an embedded
client folder you can query with no server (and what the evaluation below reads), and
`qdrant_export/` is the `.npy` + `.jsonl` + manifest that the ETL pipeline restores into a
real Qdrant server.

In [ ]:
#@title Build
ENTREZ_MAP = f"{DATA_DIR}/entrez_to_uniprots.json"
import os
entrez_arg = f"--entrez-map {ENTREZ_MAP}" if os.path.exists(ENTREZ_MAP) else ""
print("entrez map:", entrez_arg or "not present -- entrez_id will be absent from records")

!python src/builders/Qdrant/build/store.py build \
    --corpus {DATA_DIR}/eval_corpus.json \
    --model {MODEL_DIR} \
    --storage {STORE_DIR} \
    --export {EXPORT_DIR} \
    --window {WINDOW} --stride {STRIDE} \
    {entrez_arg}

In [ ]:
#@title Evaluate -- the number the ETL gate will re-check
!python src/builders/Qdrant/analysis/evaluate.py \
    --data-dir {DATA_DIR} \
    --model {MODEL_DIR} \
    --storage {STORE_DIR} \
    --out {EVAL_DIR}

import json
summary = json.load(open(f"{EVAL_DIR}/summary.json"))
score = summary["summary"]["record_hybrid"]["ndcg@10"]
print(f"\nrecord_hybrid ndcg@10 = {score:.4f}")
print("qdrant_collection_build refuses to promote a build below QDRANT_GATE_FLOOR "
      "(default 0.35). If this is under that, do not ship it.")

## 5. Package and deliver

Strip the HuggingFace checkpoint folders first. `output_dir` is the same folder the final
model saves into, and `save_total_limit=2` leaves two full ~440MB checkpoints behind — they
would triple the zip for nothing, since `load_best_model_at_end` already folded the best one
into the saved model.

In [ ]:
#@title Zip the artifacts in /content
import glob, shutil, os

# --checkpoint-dir keeps HF's per-epoch copies out of MODEL_DIR, so this is now
# a belt-and-braces sweep rather than the load-bearing step it used to be.
for ckpt in glob.glob(f"{MODEL_DIR}/checkpoint-*"):
    print("dropping stray", os.path.basename(ckpt))
    shutil.rmtree(ckpt)
print(f"model dir contents: {sorted(os.listdir(MODEL_DIR))}")

%cd /content

# -r for the tree, -q to keep the cell output readable. Each archive has exactly
# one top-level directory, which is what qdrant_build.ingest_artifacts strips.
!rm -f /content/biobert-go-retrieval.zip /content/qdrant_store.zip /content/qdrant_export.zip
!zip -r -q /content/biobert-go-retrieval.zip biobert-go-retrieval
!zip -r -q /content/qdrant_store.zip qdrant_store
!zip -r -q /content/qdrant_export.zip qdrant_export

!ls -lh /content/*.zip

In [ ]:
#@title Copy to Drive
import json, os, shutil, time

manifest = json.load(open(f"{EXPORT_DIR}/collection_config.json"))
summary = json.load(open(f"{EVAL_DIR}/summary.json"))

for name in ("biobert-go-retrieval.zip", "qdrant_store.zip", "qdrant_export.zip"):
    src, dst = f"/content/{name}", os.path.join(DRIVE_DIR, name)
    print(f"{name:28s} {os.path.getsize(src)/1e6:8.1f} MB -> {dst}")
    shutil.copy2(src, dst)

# A receipt beside the zips, so the ETL side can see what it is about to promote
# without unpacking 500MB first.
receipt = {
    "delivered": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "dataset_release": latest["tag_name"],
    "manifest": manifest,
    "record_hybrid_ndcg@10": summary["summary"]["record_hybrid"]["ndcg@10"],
}
with open(os.path.join(DRIVE_DIR, "delivery.json"), "w") as f:
    json.dump(receipt, f, indent=2)
print("\n" + json.dumps(receipt, indent=2)[:800])

## 6. Hand off to Airflow

Then, on the ETL host:

```bash
# until the Drive hop is automated (rclone or the Google provider -- see
# ingest_artifacts in dags/qdrant_build.py), copy the three zips into:
#   data/qdrant/incoming/
airflow dags trigger qdrant_collection_build
```

That DAG unpacks the zips, re-runs the evaluation as a gate, restores the export into
`function_records__v<timestamp>` / `function_chunks__v<timestamp>`, and only then flips the
aliases the MCP server queries. The previous version stays for a rollback.

The cell below triggers it from here instead, if your Airflow is reachable from Colab —
it usually is not (a localhost instance behind WSL needs a tunnel), so treat it as optional.

In [ ]:
#@title (optional) Trigger the DAG over the Airflow REST API
AIRFLOW_URL      = ""  #@param {type:"string"}
AIRFLOW_USERNAME = ""  #@param {type:"string"}
AIRFLOW_PASSWORD = ""  #@param {type:"string"}

if not AIRFLOW_URL:
    print("No AIRFLOW_URL -- trigger it by hand: airflow dags trigger qdrant_collection_build")
else:
    import requests
    # Airflow 3 issues a JWT; the stable API is /api/v2.
    token = requests.post(f"{AIRFLOW_URL}/auth/token", timeout=30,
                          json={"username": AIRFLOW_USERNAME,
                                "password": AIRFLOW_PASSWORD}).json()["access_token"]
    run = requests.post(
        f"{AIRFLOW_URL}/api/v2/dags/qdrant_collection_build/dagRuns",
        headers={"Authorization": f"Bearer {token}"}, timeout=60,
        json={"logical_date": None,
              "conf": {"dataset_release": latest["tag_name"]}})
    run.raise_for_status()
    print(run.json()["dag_run_id"], run.json()["state"])